<a href="https://colab.research.google.com/github/kunphat510214-netizen/project6/blob/Step-8/step_08_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นที่ 8: สร้างกราฟและสรุปผล

ก่อนรัน ให้วาง `coffee_orders.csv` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [ ]:
"""ขั้นที่ 8: สร้างกราฟและสรุปผลการวิเคราะห์ร้านกาแฟ."""

from pathlib import Path  # นำเข้าโมดูล Path สำหรับการจัดการพาธไฟล์

import matplotlib.pyplot as plt  # นำเข้า Matplotlib สำหรับการสร้างกราฟ
import pandas as pd  # นำเข้า Pandas สำหรับการจัดการข้อมูลในรูปแบบ DataFrame


CSV_PATH = Path("coffee_orders.csv")  # กำหนดพาธของไฟล์ CSV ที่มีข้อมูลออเดอร์กาแฟ
CHART_PATH = Path("coffee_analysis.png")  # กำหนดพาธสำหรับบันทึกไฟล์กราฟผลการวิเคราะห์

if not CSV_PATH.exists():  # ตรวจสอบว่าไฟล์ coffee_orders.csv มีอยู่หรือไม่
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน")  # ถ้าไม่พบไฟล์ ให้แสดงข้อผิดพลาด


# โหลดข้อมูลใหม่เพื่อให้ไฟล์ขั้นที่ 8 รันแยกจากขั้นที่ 7 ได้
coffee_df = pd.read_csv(CSV_PATH)  # โหลดข้อมูลจากไฟล์ CSV เข้าสู่ DataFrame

menu_summary = (  # สร้าง DataFrame สรุปข้อมูลเมนู
    coffee_df.groupby("menu_name")  # จัดกลุ่มข้อมูลตามชื่อเมนู
    .agg(  # ใช้ฟังก์ชัน aggregate เพื่อคำนวณค่าต่างๆ
        total_orders=("order_id", "count"),  # นับจำนวนออเดอร์ทั้งหมดของแต่ละเมนู
        total_revenue=("price", "sum"),  # รวมรายได้ทั้งหมดของแต่ละเมนู
        avg_wait_minutes=("wait_minutes", "mean"),  # คำนวณเวลารอเฉลี่ยของแต่ละเมนู
    )
    .reset_index()  # รีเซ็ต index เพื่อให้ 'menu_name' กลับมาเป็นคอลัมน์ปกติ
    .sort_values("total_revenue", ascending=False)  # เรียงลำดับเมนูตามรายได้รวมจากมากไปน้อย
)
top5_menu = menu_summary.head(5)  # เลือก 5 อันดับแรกของเมนูที่มีรายได้สูงสุด
payment_counts = coffee_df["payment_method"].value_counts()  # นับจำนวนออเดอร์ตามวิธีการชำระเงิน

In [ ]:
# 8.1 สร้างกราฟอย่างน้อย 3 กราฟ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # สร้างพื้นที่สำหรับกราฟ 3 กราฟในแถวเดียวกัน

top5_menu.plot.bar(  # สร้างกราฟแท่งสำหรับ 5 อันดับเมนู
    x="menu_name",  # กำหนดแกน x เป็นชื่อเมนู
    y="total_revenue",  # กำหนดแกน y เป็นรายได้รวม
    ax=axes[0],  # วางในตำแหน่งกราฟที่ 1
    legend=False,  # ไม่แสดงคำอธิบายสัญลักษณ์
    color="steelblue",  # กำหนดสีแท่งกราฟ
)
axes[0].set_title("Top 5 Menu Revenue")  # ตั้งชื่อหัวข้อกราฟที่ 1
axes[0].set_xlabel("Menu")  # ตั้งชื่อแกน x
axes[0].set_ylabel("Revenue (THB)")  # ตั้งชื่อแกน y

payment_counts.plot.bar(ax=axes[1], color="seagreen")  # สร้างกราฟแท่งจำนวนออเดอร์ตามวิธีชำระเงินในตำแหน่งที่ 2
axes[1].set_title("Orders by Payment Method")  # ตั้งชื่อหัวข้อกราฟที่ 2
axes[1].set_xlabel("Payment Method")  # ตั้งชื่อแกน x
axes[1].set_ylabel("Orders")  # ตั้งชื่อแกน y

coffee_df["wait_minutes"].plot.hist(  # สร้างกราฟฮิสโตแกรมแสดงการกระจายของเวลารอ
    bins=9,  # แบ่งช่วงข้อมูลเป็น 9 ช่วง
    ax=axes[2],  # วางในตำแหน่งกราฟที่ 3
    color="darkorange",  # กำหนดสีส้ม
    edgecolor="black",  # กำหนดสีขอบแท่ง
)
axes[2].set_title("Queue Waiting Time")  # ตั้งชื่อหัวข้อกราฟที่ 3
axes[2].set_xlabel("Minutes")  # ตั้งชื่อแกน x
axes[2].set_ylabel("Orders")  # ตั้งชื่อแกน y

plt.tight_layout()  # ปรับการวางเลย์เอาต์ของกราฟให้เหมาะสมไม่ซ้อนทับกัน
plt.savefig(CHART_PATH, dpi=150, bbox_inches="tight")  # บันทึกรูปภาพกราฟลงไฟล์
plt.show()  # แสดงกราฟออกมาบนหน้าจอ

In [ ]:
# 8.2 สรุปผลอย่างน้อย 3-5 ประโยค
best_menu = menu_summary.iloc[0]  # ดึงข้อมูลเมนูที่อยู่อันดับแรก (รายได้สูงสุด)
member_share = coffee_df["member_id"].notna().mean() * 100  # คำนวณเปอร์เซ็นต์ของลูกค้าที่เป็นสมาชิก
popular_payment = payment_counts.index[0]  # ดึงชื่อวิธีการชำระเงินที่นิยมที่สุด
average_wait = coffee_df["wait_minutes"].mean()  # คำนวณค่าเฉลี่ยของเวลารอคิว

print(  # แสดงผลสรุปข้อที่ 1 เรื่องเมนูรายได้สูงสุด
    f"1. เมนูที่สร้างรายได้สูงสุดคือ {best_menu['menu_name']} "
    f"รวม {best_menu['total_revenue']:,.2f} บาท"
)
print(f"2. ลูกค้าสมาชิกคิดเป็น {member_share:.1f}% ของออเดอร์ทั้งหมด")  # แสดงผลสรุปข้อที่ 2 เรื่องสมาชิก
print(f"3. ช่องทางชำระเงินที่ใช้มากที่สุดคือ {popular_payment}")  # แสดงผลสรุปข้อที่ 3 เรื่องการชำระเงิน
print(f"4. เวลารอเฉลี่ยอยู่ที่ {average_wait:.1f} นาที")  # แสดงผลสรุปข้อที่ 4 เรื่องเวลารอ
print("5. ร้านควรเตรียมวัตถุดิบเมนูยอดนิยมและเพิ่มกำลังคนเมื่อคิวรอนาน")  # แสดงข้อเสนอแนะเพิ่มเติม
print(f"บันทึกกราฟไว้ที่ {CHART_PATH}")  # แจ้งตำแหน่งที่บันทึกไฟล์รูปภาพ